In [355]:
# Inteligencia de Negocios

## Componente Práctico Semana 2

## Grupo # 3

### ALEJANDRA BEATRIZ TELLO GONZÁLEZ
### PABLO RAMIRO VALLEJO ZÚÑIGA

### Importación de dependencias

Se importan las librerías necesarias para:
- **pandas**: manipulación y análisis de DataFrames
- **sqlalchemy + psycopg2**: conexión ORM a PostgreSQL
- **json**: lectura del archivo de vulnerabilidades
- **dotenv**: carga de credenciales desde `.env` (buena práctica de seguridad)


In [356]:
from unittest.mock import inplace

import pandas as pd
import numpy as np
import json
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
print("Dependencias cargadas correctamente")

Dependencias cargadas correctamente


### Variables de Entorno

-- Ver sección 2. Configuración segura del proyecto--

In [357]:
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")

print(f" Conectando a: {DB_HOST}:{DB_PORT}/{DB_NAME} como usuario '{DB_USER}'")

 Conectando a: localhost:5432/dbmcib12b como usuario 'admin'


# Desarrollo
## 1. Exploración inicial de las fuentes de datos

De acuerdo al análisis realizado, las fuentes de datos seleccionadas en el Componente Práctico de la semana 1 no poseen campos relacionales que nos permitan vincularlas en una sola tabla compuesta, por lo que hemos decidido realizar una nueva selección de fuentes de datos, la cual está vinculada al análisis de ciberseguridad, y contiene información sobre los equipos de la empresa, logs de acceso y registros de tráfico de red.

El objetivo final de este ejercicio será realizar un análisis de la criticidad de las alertas de seguridad para clasificar las que representen un mayor riesgo a la organización. Así mismo podríamos realizar un análisis de los registros de login con los incidentes de seguridad para determinar una relación entre estos datos

## 1.1. Registro de Equipos Empresariales (Base de datos postgres)

Empleamos la base de datos llamada assets que contiene información sobre los dispositivos de la empresa.

### Creación de data frame desde tabla de datos postgres assets

In [358]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

df_assets = pd.read_sql("SELECT * FROM assets", engine)


In [359]:
df_assets

,device_ip,device_name,department,criticality_level
0,192.168.140.208,SRV-ELECTION-91,Finanzas,Media
1,10.52.122.63,SRV-REQUIRE-13,IT-Operaciones,Baja
2,192.168.44.131,SRV-PERFORMANCE-41,Tecnología,Baja
3,172.18.255.141,SRV-DISCOVER-27,Finanzas,Media
4,10.13.150.31,SRV-OPTION-96,Legal,Alta
...,...,...,...,...
1045,172.26.207.139,SRV-LEADER-67,RRHH,Media
1046,192.168.175.122,SRV-WEEK-97,Legal,Media
1047,172.22.7.150,SRV-COMMUNITY-54,Tecnología,Alta
1048,172.23.45.163,SRV-SHORT-80,Legal,Baja


Estructura General

In [360]:
df_assets.info()

<class 'pandas.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   device_ip          1050 non-null   str  
 1   device_name        1050 non-null   str  
 2   department         1050 non-null   str  
 3   criticality_level  1050 non-null   str  
dtypes: str(4)
memory usage: 32.9 KB


In [361]:
df_assets.isnull().any()

device_ip            False
device_name          False
department           False
criticality_level    False
dtype: bool

In [362]:
df_assets.drop_duplicates()

,device_ip,device_name,department,criticality_level
0,192.168.140.208,SRV-ELECTION-91,Finanzas,Media
1,10.52.122.63,SRV-REQUIRE-13,IT-Operaciones,Baja
2,192.168.44.131,SRV-PERFORMANCE-41,Tecnología,Baja
3,172.18.255.141,SRV-DISCOVER-27,Finanzas,Media
4,10.13.150.31,SRV-OPTION-96,Legal,Alta
...,...,...,...,...
1045,172.26.207.139,SRV-LEADER-67,RRHH,Media
1046,192.168.175.122,SRV-WEEK-97,Legal,Media
1047,172.22.7.150,SRV-COMMUNITY-54,Tecnología,Alta
1048,172.23.45.163,SRV-SHORT-80,Legal,Baja


El data frame df_assets contiene 1050 registros clasificados en 4 columnas, que nos dan datos sobre la dirección IP del equipo, el nombre del mismo, el departamento al que pertenece y el nivel de criticidad de dicho equipo.

Todas las columnas en este data frame son de tipo string y no poseen valores nulos ni duplicados que limpiar.

De este data frame nos parecen de particular importancia las columnas device_ip y department para ser relacionadas con las otras fuentes de datos.

## 1.2. Registros de tráfico de red (Fuente de Datos CSV)

### Estructura General

El segundo data frame contiene logs del tráfico de red. No tenemos valores nulos por limpiar.

In [363]:
df_network = pd.read_csv('data/network_traffic_large.csv')

df_network

,Source_IP,Destination_IP,Timestamp,Protocol,Total_Packets,Label
0,172.22.78.28,219.187.29.255,13/06/2026 08:00:32,ICMP,11028,Brute Force
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,TCP,178,Benign
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,TCP,41961,Port Scan
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,TCP,39784,DoS
4,192.168.78.253,204.98.57.12,2026-06-13 08:02:40,ICMP,3699,Botnet
...,...,...,...,...,...,...
1195,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,TCP,409,Benign
1196,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,TCP,213,Benign
1197,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,ICMP,14573,DoS
1198,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,TCP,183,Benign


El data frame de red contiene 5 columnas (Source_IP, Destination_IP, Timestamp, Protocol, Total_Packets y Label) y 1200 filas. La columna Total_Packets es de tipo entero, las demás son todas de tipo string. Destacamos que dentro del campo "Timestamp" existen dos tipos de formato que deberán ser unificados.

De este data frame se puede notar que la columna "Source_IP" podría ser de importancia al momento de unificar las tablas de datos.

In [364]:
df_network.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Source_IP       1200 non-null   str  
 1   Destination_IP  1200 non-null   str  
 2   Timestamp       1200 non-null   str  
 3   Protocol        1200 non-null   str  
 4   Total_Packets   1200 non-null   int64
 5   Label           1200 non-null   str  
dtypes: int64(1), str(5)
memory usage: 56.4 KB


## 1.3. Registros de Autenticación (Fuente de Datos json)


El tercer data frame contiene datos de logs de autenticación, teniendo un total de 1300 registros, con 5 columnas de datos (event_id, host_ip, user_id, status y time). No tenemos valores nulos por limpiar.

In [365]:
df_auth = pd.read_json('data/auth_logs_large.json')

df_auth

,event_id,host_ip,user_id,status,time
0,10000,10.53.158.133,shermandonna,Success,2026-06-13T07:55:18
1,10001,172.29.167.220,yschmidt,Failed,2026-06-13T07:55:37
2,10002,10.71.212.242,rgonzalez,Success,2026-06-13T07:55:59
3,10003,172.23.233.99,kevin46,Failed,2026-06-13T07:56:06
4,10004,172.17.198.40,veronica90,Failed,2026-06-13T07:56:24
...,...,...,...,...,...
1295,11295,10.39.221.155,ynichols,Success,2026-06-13T16:35:46
1296,11296,10.66.250.203,tammykim,Success,2026-06-13T16:36:09
1297,11297,172.18.54.140,jmorrow,Success,2026-06-13T16:36:34
1298,11298,192.168.47.213,christiancontreras,Success,2026-06-13T16:37:17


La columna "event_id" es de tipo entero, las demás son de tipo string. La columna "host_ip" nos resulta relevante para la posterior unificación de los data frames.

In [366]:
df_auth.info()

<class 'pandas.DataFrame'>
RangeIndex: 1300 entries, 0 to 1299
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   event_id  1300 non-null   int64
 1   host_ip   1300 non-null   str  
 2   user_id   1300 non-null   str  
 3   status    1300 non-null   str  
 4   time      1300 non-null   str  
dtypes: int64(1), str(4)
memory usage: 50.9 KB


Eliminamos posibles registros duplicados.

In [367]:
df_auth.drop_duplicates()

,event_id,host_ip,user_id,status,time
0,10000,10.53.158.133,shermandonna,Success,2026-06-13T07:55:18
1,10001,172.29.167.220,yschmidt,Failed,2026-06-13T07:55:37
2,10002,10.71.212.242,rgonzalez,Success,2026-06-13T07:55:59
3,10003,172.23.233.99,kevin46,Failed,2026-06-13T07:56:06
4,10004,172.17.198.40,veronica90,Failed,2026-06-13T07:56:24
...,...,...,...,...,...
1295,11295,10.39.221.155,ynichols,Success,2026-06-13T16:35:46
1296,11296,10.66.250.203,tammykim,Success,2026-06-13T16:36:09
1297,11297,172.18.54.140,jmorrow,Success,2026-06-13T16:36:34
1298,11298,192.168.47.213,christiancontreras,Success,2026-06-13T16:37:17


# 2. Configuración segura del proyecto

### Manejo de variables de entorno

Las credenciales de la base de datos se leen del archivo `.env` para evitar exponer datos sensibles en el código fuente.
Esta práctica es recomendable dentro de ambientes tanto de pruebas como de producción para no exponer las credenciales en los códigos fuente, que podrían ser publicadas de forma accidental al subir los archivos del proyecto a repositorios públicos como GitHub. Existen bots automatizados rastreando repositorios públicos constantemente en busca de contraseñas y claves de AWS, Firebase o bases de datos. Si encuentran una, podrían usar tus servicios, generar deudas enormes o robar tu información en cuestión de minutos.

También debemos tomar en cuenta que, al publicar los proyectos en repositorios como GitHub, los archivos .env sean excluídos de la carga mediante la creación de los archivos .gitignore.

Se emplearon las variables DB_USER, DB_PASSWORD, DB_NAME, DB_HOST Y DB_PORT.

Esta práctica aporta también con la separación de entornos de desarrollo, pruebas y producción, una aplicación no se comportará igual ni usará los mismos recursos cuando estás programando en tu computadora que cuando ya está publicada en internet.

En desarrollo: Usas una base de datos local (localhost) con datos de prueba ficticios.

En producción: Usas la base de datos real con los datos de tus usuarios.

Al usar variables de entorno, el código fuente es exactamente el mismo para todos los casos. Lo único que cambia es el archivo .env que está instalado en el servidor de producción o en tu máquina local. No tienes que modificar ni una sola línea de código para cambiar de entorno.

Así mismo es mucho más sencillo actualizar la información de tokens o claves de acceso, ya que solo debemos modificar el archivo .env en lugar de actualizar dicha información en todos los archivos que la requieren.



# 3. Limpieza de Datos

## 3.1 Unificación de formatos de tiempo

Dentro del data frame "df_network", en la columna "Timestamp", se presentan registros cuyo formato de fecha difiere del resto de registros, por lo que usaremos la función "pd.to_datetime", que nos permite convertir las fechas que se encuentran en otro formato al formato estándar de tiempo.

El argumento "errors="coerce"" le indica a la función que si encuentra formatos incomprensibles los convierta en NaT(Not a Time).

El argumento "dayfirst" le indica a la función que el formato de fechas utiliza el valor del día antes del mes, formato usado mayormente el latinoamerica que difiere del formato utilizado en Estados Unidos donde el mes se coloca antes que el día.

El argumento "format", le indica a la función que va a encontrar distintos formatos dentro de los registros de datos y los tratará de forma independiente.


In [368]:
df_network["Timestamp"] = pd.to_datetime(df_network["Timestamp"], errors="coerce", dayfirst=True, format='mixed')

df_network

,Source_IP,Destination_IP,Timestamp,Protocol,Total_Packets,Label
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,ICMP,11028,Brute Force
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,TCP,178,Benign
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,TCP,41961,Port Scan
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,TCP,39784,DoS
4,192.168.78.253,204.98.57.12,2026-06-13 08:02:40,ICMP,3699,Botnet
...,...,...,...,...,...,...
1195,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,TCP,409,Benign
1196,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,TCP,213,Benign
1197,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,ICMP,14573,DoS
1198,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,TCP,183,Benign


Adicionalmente esta función convierte el tipo de datos de la columna "Timestamp", a "datetime64[us]", que nos va a permitir usar funciones ".dt" y extraer información como el año, mes o día de forma directa.

In [369]:
df_network.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Source_IP       1200 non-null   str           
 1   Destination_IP  1200 non-null   str           
 2   Timestamp       1200 non-null   datetime64[us]
 3   Protocol        1200 non-null   str           
 4   Total_Packets   1200 non-null   int64         
 5   Label           1200 non-null   str           
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 56.4 KB


In [370]:
df_network["Timestamp"].dt.month

0       6
1       6
2       6
3       6
4       6
       ..
1195    6
1196    6
1197    6
1198    6
1199    6
Name: Timestamp, Length: 1200, dtype: int32

Repetimos la función para el data frame "df_auth"

In [371]:
df_auth["time"] = pd.to_datetime(df_auth["time"], errors="coerce", dayfirst=True, format='mixed')

df_auth


,event_id,host_ip,user_id,status,time
0,10000,10.53.158.133,shermandonna,Success,2026-06-13 07:55:18
1,10001,172.29.167.220,yschmidt,Failed,2026-06-13 07:55:37
2,10002,10.71.212.242,rgonzalez,Success,2026-06-13 07:55:59
3,10003,172.23.233.99,kevin46,Failed,2026-06-13 07:56:06
4,10004,172.17.198.40,veronica90,Failed,2026-06-13 07:56:24
...,...,...,...,...,...
1295,11295,10.39.221.155,ynichols,Success,2026-06-13 16:35:46
1296,11296,10.66.250.203,tammykim,Success,2026-06-13 16:36:09
1297,11297,172.18.54.140,jmorrow,Success,2026-06-13 16:36:34
1298,11298,192.168.47.213,christiancontreras,Success,2026-06-13 16:37:17


In [372]:
df_auth.info()

<class 'pandas.DataFrame'>
RangeIndex: 1300 entries, 0 to 1299
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   event_id  1300 non-null   int64         
 1   host_ip   1300 non-null   str           
 2   user_id   1300 non-null   str           
 3   status    1300 non-null   str           
 4   time      1300 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 50.9 KB


## 3.2 Renombrar columnas para fácil identificación

Una de las columnas que nos enlaza las tablas de datos entre sí podría ser la columna "Source_IP" del data frame "df_network", la cual podría relacionarse con la columna "host_ip" del data frame "df_auth", por lo que consideramos importante que ambas columnas lleven el mismo nombre.

También procederemos a renombrar la columna "time" del data frame "df_auth", por "Auth_Timestamp" para poder identificarse de mejor manera el momento de unificar las tablas.

In [373]:
df_auth = df_auth.rename(columns={"host_ip": "Source_IP","time": "Auth_Timestamp"})

df_auth

,event_id,Source_IP,user_id,status,Auth_Timestamp
0,10000,10.53.158.133,shermandonna,Success,2026-06-13 07:55:18
1,10001,172.29.167.220,yschmidt,Failed,2026-06-13 07:55:37
2,10002,10.71.212.242,rgonzalez,Success,2026-06-13 07:55:59
3,10003,172.23.233.99,kevin46,Failed,2026-06-13 07:56:06
4,10004,172.17.198.40,veronica90,Failed,2026-06-13 07:56:24
...,...,...,...,...,...
1295,11295,10.39.221.155,ynichols,Success,2026-06-13 16:35:46
1296,11296,10.66.250.203,tammykim,Success,2026-06-13 16:36:09
1297,11297,172.18.54.140,jmorrow,Success,2026-06-13 16:36:34
1298,11298,192.168.47.213,christiancontreras,Success,2026-06-13 16:37:17


Procedemos de la misma manera con la columna "Timestamp" será renombrada a "Network_Timestamp" para una mejor identificación.

In [374]:
df_network = df_network.rename(columns={"Timestamp": "Network_Timestamp"})

df_network

,Source_IP,Destination_IP,Network_Timestamp,Protocol,Total_Packets,Label
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,ICMP,11028,Brute Force
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,TCP,178,Benign
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,TCP,41961,Port Scan
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,TCP,39784,DoS
4,192.168.78.253,204.98.57.12,2026-06-13 08:02:40,ICMP,3699,Botnet
...,...,...,...,...,...,...
1195,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,TCP,409,Benign
1196,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,TCP,213,Benign
1197,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,ICMP,14573,DoS
1198,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,TCP,183,Benign


# 4. Selección de columnas relevantes





### 4.1 Data frame "df_assets"

Del data frame "df_assets" se conservarán todas las columnas ya que podrían resultar útiles para nuestro análisis.

In [375]:
df_assets.info()

<class 'pandas.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   device_ip          1050 non-null   str  
 1   device_name        1050 non-null   str  
 2   department         1050 non-null   str  
 3   criticality_level  1050 non-null   str  
dtypes: str(4)
memory usage: 32.9 KB


### 4.2 Data frame "df_network"

Del data frame "df_network", eliminaremos las columnas "Protocol" y "Total_Packets", que no resultan de importancia particular en el objetivo final de nuestro análisis.

Emplearemos la función ".drop" y guardaremos el data frame resultante en un nuevo data frame llamado "df_network_cut"


In [376]:
df_network_cut = df_network.drop(['Protocol','Total_Packets'], axis=1)

In [377]:
df_network_cut

,Source_IP,Destination_IP,Network_Timestamp,Label
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS
4,192.168.78.253,204.98.57.12,2026-06-13 08:02:40,Botnet
...,...,...,...,...
1195,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign
1196,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign
1197,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS
1198,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign


In [378]:
df_network_cut.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Source_IP          1200 non-null   str           
 1   Destination_IP     1200 non-null   str           
 2   Network_Timestamp  1200 non-null   datetime64[us]
 3   Label              1200 non-null   str           
dtypes: datetime64[us](1), str(3)
memory usage: 37.6 KB


### 4.3 Data frame "df_auth"

Del data frame "df_auth" se conservarán las columnas "Source_IP", "user_id", "status" y "Auth_Timestamp" que podrían mantener relación con los objetivos del análisis.

Eliminaremos la columna "event_id" que no nos brinda información útil para el análisis, siendo quizás necesario generar un nuevo identificador para los datos resultantes de la unificación de tablas.

Igual que en la data frame anterior guardaremos el resultado de la función drop en un nuevo data frame sin alterar el original.

In [379]:
df_auth_cut = df_auth.drop(['event_id'], axis=1)

df_auth_cut.info()

<class 'pandas.DataFrame'>
RangeIndex: 1300 entries, 0 to 1299
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Source_IP       1300 non-null   str           
 1   user_id         1300 non-null   str           
 2   status          1300 non-null   str           
 3   Auth_Timestamp  1300 non-null   datetime64[us]
dtypes: datetime64[us](1), str(3)
memory usage: 40.8 KB


### 4.4 Columnas para relación de datos

Para relacionar los datos de las tablas principalmente se emplearán las columnas que contienen la dirección IP de los equipos, esto es "device_ip" del data frame "assets" y "Source_IP" de los data frames "df_network_cut" y "df_auth_cut"

# 5. Transformaciones Requeridas

### 5.1 Procesamiento de datos en la columna "device_name"

Dentro del data frame "assets"se tiene la columna "device_name", que en todos los casos inicia con el string "SRV-", el cual puede ser limpiado para mejorar la comprensión e identificación de esta columa.


In [380]:
df_assets['device_name'] = df_assets['device_name'].str.replace('SRV-', '', regex=False)

df_assets

,device_ip,device_name,department,criticality_level
0,192.168.140.208,ELECTION-91,Finanzas,Media
1,10.52.122.63,REQUIRE-13,IT-Operaciones,Baja
2,192.168.44.131,PERFORMANCE-41,Tecnología,Baja
3,172.18.255.141,DISCOVER-27,Finanzas,Media
4,10.13.150.31,OPTION-96,Legal,Alta
...,...,...,...,...
1045,172.26.207.139,LEADER-67,RRHH,Media
1046,192.168.175.122,WEEK-97,Legal,Media
1047,172.22.7.150,COMMUNITY-54,Tecnología,Alta
1048,172.23.45.163,SHORT-80,Legal,Baja


### 5.2 Unificación de tablas

Unificamos las tablas del inventario de equipos con los logs de tráfico de red. Emplearemos la función "merge", mediante las columnas "Source_IP" y "device_ip"

In [381]:
df_join_assets = pd.merge(df_network_cut, df_assets, left_on='Source_IP', right_on='device_ip', how='left').drop(columns='device_ip')

df_join_assets

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja
4,192.168.78.253,204.98.57.12,2026-06-13 08:02:40,Botnet,SISTER-23,RRHH,Media
...,...,...,...,...,...,...,...
1195,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media
1196,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media
1197,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media
1198,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja


Del resultado encontramos que se han generado valores nulos en las columnas "device_name", "department" y "criticality_level", de lo que podemos deducir que se trata de incidentes generados por equipos externos a la empresa, ya que sus direcciones ip no se encuentran registradas en el inventario.

Para tratar estos datos no sería adecuado eliminarlos, ya que al ser ataques externos siempre debemos realizar un análisis de los mismos, una mejor solución sería colocar estos valores como "Externo" y siempre en en un nivel de criticidad "Alta". Para ello usaremos la función ".fillna"


In [382]:
df_join_assets.fillna({'device_name': 'External'}, inplace=True)

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja
4,192.168.78.253,204.98.57.12,2026-06-13 08:02:40,Botnet,SISTER-23,RRHH,Media
...,...,...,...,...,...,...,...
1195,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media
1196,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media
1197,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media
1198,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja


Repetimos la operación para la columna "department"

In [383]:
df_join_assets.fillna({'department': 'External'}, inplace=True)

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja
4,192.168.78.253,204.98.57.12,2026-06-13 08:02:40,Botnet,SISTER-23,RRHH,Media
...,...,...,...,...,...,...,...
1195,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media
1196,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media
1197,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media
1198,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja


Repetimos la operación para la columna "criticality_level"

In [384]:
df_join_assets.fillna({'criticality_level': 'Alta'}, inplace=True)

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja
4,192.168.78.253,204.98.57.12,2026-06-13 08:02:40,Botnet,SISTER-23,RRHH,Media
...,...,...,...,...,...,...,...
1195,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media
1196,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media
1197,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media
1198,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja


Ahora incorporamos la tabla de registros de autenticación a la tabla "df_join_assets"

In [385]:
df_auth_cut

,Source_IP,user_id,status,Auth_Timestamp
0,10.53.158.133,shermandonna,Success,2026-06-13 07:55:18
1,172.29.167.220,yschmidt,Failed,2026-06-13 07:55:37
2,10.71.212.242,rgonzalez,Success,2026-06-13 07:55:59
3,172.23.233.99,kevin46,Failed,2026-06-13 07:56:06
4,172.17.198.40,veronica90,Failed,2026-06-13 07:56:24
...,...,...,...,...
1295,10.39.221.155,ynichols,Success,2026-06-13 16:35:46
1296,10.66.250.203,tammykim,Success,2026-06-13 16:36:09
1297,172.18.54.140,jmorrow,Success,2026-06-13 16:36:34
1298,192.168.47.213,christiancontreras,Success,2026-06-13 16:37:17


In [386]:
df_compsed = pd.merge(df_join_assets, df_auth_cut, on='Source_IP', how='left')

df_compsed

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level,user_id,status,Auth_Timestamp
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja,NaN,NaN,NaT
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media,NaN,NaN,NaT
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media,regina31,Success,2026-06-13 13:32:45
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,jimmy80,Success,2026-06-13 11:30:34
4,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,lance10,Success,2026-06-13 12:30:05
...,...,...,...,...,...,...,...,...,...,...
1752,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media,jennifer66,Success,2026-06-13 11:28:15
1753,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media,NaN,NaN,NaT
1754,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media,sarah75,Success,2026-06-13 14:11:38
1755,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja,blairjill,Success,2026-06-13 09:01:20


Esta combinación nos resulta en la generación de algunos valores nulos en los campos "user_id", "status" y "Auth_Timestamp", lo que indicaría que son incidentes de red no relacionados con inicios de sesión, lo que podría solventarse insertando valores para rellenar estos campos nulos o eliminando estos registros, dependiento del tipo de análisis al que querramos llegar. Como consideramos que se puede tratar de información relevante para los procesos de análisis de los datos, procederemos a inputar datos 'No relacionado a inicio de sesion' en los valores nulos, para las  columnas con valores faltantes, y para la columna 'Auth_Timestamp', copiaremos el Timestamp de los logs de red.

In [387]:
df_compsed.fillna({'user_id': 'No relacionado a inicio de sesión'}, inplace=True)

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level,user_id,status,Auth_Timestamp
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja,No relacionado a inicio de sesión,NaN,NaT
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media,No relacionado a inicio de sesión,NaN,NaT
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media,regina31,Success,2026-06-13 13:32:45
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,jimmy80,Success,2026-06-13 11:30:34
4,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,lance10,Success,2026-06-13 12:30:05
...,...,...,...,...,...,...,...,...,...,...
1752,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media,jennifer66,Success,2026-06-13 11:28:15
1753,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media,No relacionado a inicio de sesión,NaN,NaT
1754,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media,sarah75,Success,2026-06-13 14:11:38
1755,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja,blairjill,Success,2026-06-13 09:01:20


In [388]:
df_compsed.fillna({'status': 'No aplica'}, inplace=True)

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level,user_id,status,Auth_Timestamp
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja,No relacionado a inicio de sesión,No aplica,NaT
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media,No relacionado a inicio de sesión,No aplica,NaT
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media,regina31,Success,2026-06-13 13:32:45
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,jimmy80,Success,2026-06-13 11:30:34
4,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,lance10,Success,2026-06-13 12:30:05
...,...,...,...,...,...,...,...,...,...,...
1752,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media,jennifer66,Success,2026-06-13 11:28:15
1753,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media,No relacionado a inicio de sesión,No aplica,NaT
1754,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media,sarah75,Success,2026-06-13 14:11:38
1755,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja,blairjill,Success,2026-06-13 09:01:20


In [389]:
df_compsed.fillna({'Auth_Timestamp': df_compsed['Network_Timestamp']}, inplace=True)

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level,user_id,status,Auth_Timestamp
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:32
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:56
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media,regina31,Success,2026-06-13 13:32:45
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,jimmy80,Success,2026-06-13 11:30:34
4,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,lance10,Success,2026-06-13 12:30:05
...,...,...,...,...,...,...,...,...,...,...
1752,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media,jennifer66,Success,2026-06-13 11:28:15
1753,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 18:49:01
1754,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media,sarah75,Success,2026-06-13 14:11:38
1755,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja,blairjill,Success,2026-06-13 09:01:20


### 5.3 Creación de columnas nuevas

Para este análisis consideramos pertinente realizar la creación de una nueva columna que nos permita evaluar la prioridad con la que debemos atender los incidentes de red, basados en la etiqueta de la amenaza presentada y el el nivel de criticidad del activo atacado. Para inicializar el valor de esta columna establecemos el valor inicial a 'Baja'

In [390]:
df_compsed_complete = df_compsed.assign(Incident_priority = 'Baja')

df_compsed_complete




,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level,user_id,status,Auth_Timestamp,Incident_priority
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:32,Baja
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:56,Baja
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media,regina31,Success,2026-06-13 13:32:45,Baja
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,jimmy80,Success,2026-06-13 11:30:34,Baja
4,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,lance10,Success,2026-06-13 12:30:05,Baja
...,...,...,...,...,...,...,...,...,...,...,...
1752,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media,jennifer66,Success,2026-06-13 11:28:15,Baja
1753,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 18:49:01,Baja
1754,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media,sarah75,Success,2026-06-13 14:11:38,Baja
1755,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja,blairjill,Success,2026-06-13 09:01:20,Baja


Realizamos una clasificación para los elementos que poseen el nivel de criticalidad del equipo en "Alta" y que su etiqueta en el campo "Label" es diferente de "Bening", para clasificarlos como prioridad crítica. Para esto emplearemos la función ".loc"

In [391]:
df_compsed_complete["Incident_Priority"] = np.where(
    (df_compsed_complete["criticality_level"] == "Alta")
    & (df_compsed_complete["Label"] != "Benign"),
    "CRÍTICA",  # Valor si es VERDADERO
    "Baja",  # Valor si es FALSO (evita que quede nulo)
)

df_compsed_complete


,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level,user_id,status,Auth_Timestamp,Incident_priority,Incident_Priority
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:32,Baja,Baja
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:56,Baja,Baja
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media,regina31,Success,2026-06-13 13:32:45,Baja,Baja
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,jimmy80,Success,2026-06-13 11:30:34,Baja,Baja
4,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,lance10,Success,2026-06-13 12:30:05,Baja,Baja
...,...,...,...,...,...,...,...,...,...,...,...,...
1752,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media,jennifer66,Success,2026-06-13 11:28:15,Baja,Baja
1753,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 18:49:01,Baja,Baja
1754,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media,sarah75,Success,2026-06-13 14:11:38,Baja,Baja
1755,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja,blairjill,Success,2026-06-13 09:01:20,Baja,Baja


Realizamos otra clasificación de nivel de prioridad para los casos en los que si bien el vector de ataque no es benigno, pero la criticidad del activo es baja y lo clasificaremos como prioridad Media.

In [392]:
df_compsed_complete["Incident_Priority"] = np.where(
    (df_compsed_complete["criticality_level"] == "Baja")
    & (df_compsed_complete["Label"] != "Benign"),
    "Media",  # Valor si es VERDADERO
    df_compsed_complete["Incident_Priority"],
)

df_compsed_complete

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level,user_id,status,Auth_Timestamp,Incident_priority,Incident_Priority
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:32,Baja,Media
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:56,Baja,Baja
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media,regina31,Success,2026-06-13 13:32:45,Baja,Baja
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,jimmy80,Success,2026-06-13 11:30:34,Baja,Media
4,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,lance10,Success,2026-06-13 12:30:05,Baja,Media
...,...,...,...,...,...,...,...,...,...,...,...,...
1752,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media,jennifer66,Success,2026-06-13 11:28:15,Baja,Baja
1753,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 18:49:01,Baja,Baja
1754,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media,sarah75,Success,2026-06-13 14:11:38,Baja,Baja
1755,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja,blairjill,Success,2026-06-13 09:01:20,Baja,Baja


Para los eventos de autenticación fallidos combinados con anomalías en la red podemos clasificarlos como prioridad "Alta"

In [394]:
df_compsed_complete["Incident_Priority"] = np.where(
    (df_compsed_complete["status"] == "Failed")
    & (df_compsed_complete["Label"] != "Benign"),
    "Alta",  # Valor si es VERDADERO
    df_compsed_complete["Incident_Priority"],
)

df_compsed_complete

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level,user_id,status,Auth_Timestamp,Incident_priority,Incident_Priority
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:32,Baja,Media
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:56,Baja,Baja
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media,regina31,Success,2026-06-13 13:32:45,Baja,Baja
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,jimmy80,Success,2026-06-13 11:30:34,Baja,Media
4,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,lance10,Success,2026-06-13 12:30:05,Baja,Media
...,...,...,...,...,...,...,...,...,...,...,...,...
1752,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media,jennifer66,Success,2026-06-13 11:28:15,Baja,Baja
1753,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 18:49:01,Baja,Baja
1754,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media,sarah75,Success,2026-06-13 14:11:38,Baja,Baja
1755,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja,blairjill,Success,2026-06-13 09:01:20,Baja,Baja


# 6. Generación de Identificadores

Al momento no se ha considerado generar identificadores adicionales para la tabla compuesta, ya que no se requieren en el contexto de nuestro análisis.

# 7. Validación de Resultados

## 7.1 Vista Previa del DataFrame Transformado

In [395]:
df_compsed_complete

,Source_IP,Destination_IP,Network_Timestamp,Label,device_name,department,criticality_level,user_id,status,Auth_Timestamp,Incident_priority,Incident_Priority
0,172.22.78.28,219.187.29.255,2026-06-13 08:00:32,Brute Force,HIMSELF-37,Legal,Baja,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:32,Baja,Media
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,Benign,RECEIVE-56,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 08:00:56,Baja,Baja
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,Port Scan,CARD-20,RRHH,Media,regina31,Success,2026-06-13 13:32:45,Baja,Baja
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,jimmy80,Success,2026-06-13 11:30:34,Baja,Media
4,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,DoS,DARK-74,RRHH,Baja,lance10,Success,2026-06-13 12:30:05,Baja,Media
...,...,...,...,...,...,...,...,...,...,...,...,...
1752,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,Benign,AT-36,Tecnología,Media,jennifer66,Success,2026-06-13 11:28:15,Baja,Baja
1753,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,Benign,TRIP-93,Finanzas,Media,No relacionado a inicio de sesión,No aplica,2026-06-13 18:49:01,Baja,Baja
1754,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,DoS,READY-52,Tecnología,Media,sarah75,Success,2026-06-13 14:11:38,Baja,Baja
1755,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,Benign,BELIEVE-72,Tecnología,Baja,blairjill,Success,2026-06-13 09:01:20,Baja,Baja


In [396]:
df_compsed_complete.info()

<class 'pandas.DataFrame'>
RangeIndex: 1757 entries, 0 to 1756
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Source_IP          1757 non-null   str           
 1   Destination_IP     1757 non-null   str           
 2   Network_Timestamp  1757 non-null   datetime64[us]
 3   Label              1757 non-null   str           
 4   device_name        1757 non-null   str           
 5   department         1757 non-null   str           
 6   criticality_level  1757 non-null   str           
 7   user_id            1757 non-null   str           
 8   status             1757 non-null   str           
 9   Auth_Timestamp     1757 non-null   datetime64[us]
 10  Incident_priority  1757 non-null   str           
 11  Incident_Priority  1757 non-null   str           
dtypes: datetime64[us](2), str(10)
memory usage: 164.8 KB


## 7.2. Verificación de valores nulos resultantes

La tabla resultante no posee valores nulos en su forma consolidada.

In [398]:
df_compsed_complete.isnull().any()

Source_IP            False
Destination_IP       False
Network_Timestamp    False
Label                False
device_name          False
department           False
criticality_level    False
user_id              False
status               False
Auth_Timestamp       False
Incident_priority    False
Incident_Priority    False
dtype: bool

## 7.3 Revisión de Duplicados

Se procede a revisar si existen valores duplicados en las filas y se determina que, no existen valores duplicados en las filas.

In [399]:
df_compsed_complete.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
1752    False
1753    False
1754    False
1755    False
1756    False
Length: 1757, dtype: bool

## 7.4 Validación de tipos de datos

Se procede con la validación de tipos de datos de donde se obtiene que las columnas son de tipo string, excepto por las marcas de tiempo "Timestamp" y "AuthTimestamp"

In [400]:
df_compsed_complete.info()

<class 'pandas.DataFrame'>
RangeIndex: 1757 entries, 0 to 1756
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Source_IP          1757 non-null   str           
 1   Destination_IP     1757 non-null   str           
 2   Network_Timestamp  1757 non-null   datetime64[us]
 3   Label              1757 non-null   str           
 4   device_name        1757 non-null   str           
 5   department         1757 non-null   str           
 6   criticality_level  1757 non-null   str           
 7   user_id            1757 non-null   str           
 8   status             1757 non-null   str           
 9   Auth_Timestamp     1757 non-null   datetime64[us]
 10  Incident_priority  1757 non-null   str           
 11  Incident_Priority  1757 non-null   str           
dtypes: datetime64[us](2), str(10)
memory usage: 164.8 KB


# 8. Reflexión Individual Final

### ALEJANDRA BEATRIZ TELLO GONZÁLEZ




### PABLO RAMIRO VALLEJO ZÚÑIGA


